In [22]:
import os, contextlib, shutil
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

## Tabu Search Implementaton
Minimises $\sum_{}^{j}T_j$ where $T_j = max(0, C_j-d_j)$

Solves a $1|prec|\sum_{}^{j}T_j$ problem with rejection method and penalty function method.

#### Constraint Handling Methods:
1. **Rejection Method:** Filters out candidate schedules that violate precedence relations.
1. **Penalty Function Method:** Allows infeasible schedules during search by adding a penalty cost based on position displacement: 
$$ g'(S) = g(S) + \sum_{k=1}^{m}\lambda_k max(0,k_k(S)), g(S)=\sum_j T_j $$
$$ h_k(S) = \text{pos}_S(u) - \text{pos}_S(v) $$

**Input:** DAG (in adjacency list)

**Output:** Schedule (ordered list)

In [23]:
from tabu_search import tabu, preprocessing, check_precedence, tardiness

### Test Cases

In [24]:
prec = [(0, 1), (0, 2), (0, 4), (0, 5), (0, 6), (1, 7), (1, 8), (2, 3), (2, 9), 
         (3, 10), (4, 11), (5, 12), (6, 13), (7, 14), (8, 14), (9, 15), (10, 19), 
         (11, 16), (12, 17), (13, 20), (14, 18), (15, 19), (16, 26), (17, 20), 
         (18, 24), (19, 21), (20, 21), (20, 22), (21, 23), (21, 25), (22, 25), 
         (23, 27), (24, 26), (25, 26), (26, 28), (27, 29), (28, 29), (29, 30)
]

prec = [(u + 1, v + 1) for u, v in prec]   # jobs numbered 1..31

p = [2, 6, 14, 3, 3, 5, 2, 3, 2, 3, 2, 3, 6, 2, 10, 2, 2, 5, 18, 6, 14, 6, 2, 5, 5, 6, 5, 5, 10, 3, 10]
d = [329, 307, 340, 122, 73, 287, 61, 164, 256, 181, 88, 279, 290, 18, 217, 71, 307, 295, 209, 77, 217, 233, 71, 355, 141, 34, 93, 253, 82, 172, 269]

# initial solution forced by question 3.1
x0 = [1, 2, 3, 6, 13, 5, 12, 17, 7, 4, 10, 14, 9, 8, 18, 21, 11, 16, 15, 20, 19, 22, 24, 23, 25, 26, 27, 28, 29, 30, 31]

if __name__ == "__main__":
    # TEST: question 3.1 with K = 10
    tabu(p, d, prec, K=10, gamma=10, lmbda=0, L=20, x0=x0, method='rejection', verbose=True)

k=0  x=[1, 2, 3, 6, 13, 5, 12, 17, 7, 4, 10, 14, 9, 8, 18, 21, 11, 16, 15, 20, 19, 22, 24, 23, 25, 26, 27, 28, 29, 30, 31]  cost=341  valid=True  g_best=341
    cand pos 1-2 swap (1, 2): violates precedence -> skipped
    cand pos 2-3 swap (2, 3): [1, 3, 2, 6, 13, 5, 12, 17, 7, 4, 10, 14, 9, 8, 18, 21, 11, 16, 15, 20, 19, 22, 24, 23, 25, 26, 27, 28, 29, 30, 31] cost=341 delta=0 tabu=False -> ACCEPT
k=1  x=[1, 3, 2, 6, 13, 5, 12, 17, 7, 4, 10, 14, 9, 8, 18, 21, 11, 16, 15, 20, 19, 22, 24, 23, 25, 26, 27, 28, 29, 30, 31]  cost=341  valid=True  g_best=341  T=[(2, 3)]
    cand pos 3-4 swap (2, 6): [1, 3, 6, 2, 13, 5, 12, 17, 7, 4, 10, 14, 9, 8, 18, 21, 11, 16, 15, 20, 19, 22, 24, 23, 25, 26, 27, 28, 29, 30, 31] cost=341 delta=0 tabu=False -> ACCEPT
k=2  x=[1, 3, 6, 2, 13, 5, 12, 17, 7, 4, 10, 14, 9, 8, 18, 21, 11, 16, 15, 20, 19, 22, 24, 23, 25, 26, 27, 28, 29, 30, 31]  cost=341  valid=True  g_best=341  T=[(2, 3), (2, 6)]
    cand pos 4-5 swap (2, 13): [1, 3, 6, 13, 2, 5, 12, 17, 7, 4, 10,

In [25]:
OUT = "outputs"     
RES = "results"    
for folder in (OUT, RES):
    shutil.rmtree(folder, ignore_errors=True)   # start clean
    os.makedirs(folder)

# runs in results/ 
EXAMPLES_32 = [(0, 20), (100, 1), (10, 100)]   # (gamma, L) traces used in the 3.2 discussion
PLOT_LAMBDAS = (10, 20, 50)                     # lambdas shown in the 3.3 plot
EDD_LAMBDA = 25                                 # lambda reported for the EDD start
_, _, job_info = preprocessing(len(p), prec, p, d)
fmt = lambda x: ",".join(map(str, x))

def run(fname=None, **kw):
    """run tabu()
    opt: if fname is given, write full per-iteration printout to outputs/fname
    """
    if fname is None:
        return tabu(p, d, prec, 1000, kw["gamma"], kw.get("lmbda", 0), kw["L"], x0=kw["x0"],
                    method=kw.get("method", "rejection"))
    path = os.path.join(OUT, fname)
    os.makedirs(os.path.dirname(path), exist_ok=True)   # allows sub-folders, e.g. outputs/3_2_grid/
    with open(path, "w") as fh, contextlib.redirect_stdout(fh):
        # header so every trace file says which run it is
        print(f"# method={kw.get('method', 'rejection')}  K={kw.get('K', 1000)}  gamma={kw['gamma']}  "
              f"L={kw['L']}  lambda={kw.get('lmbda', 0) if kw.get('method') == 'penalty' else '-'}  "
              f"x0=[{fmt(kw['x0'])}]")
        return tabu(p, d, prec, kw.get("K", 1000), kw["gamma"], kw.get("lmbda", 0), kw["L"],
                    x0=kw["x0"], method=kw.get("method", "rejection"), verbose=True)

summary = open(os.path.join(OUT, "summary.txt"), "w")
def out(s=""):
    print(s); summary.write(s + "\n")

## 3.1

In [26]:
out("=== 3.1 Rejection method, L=20, gamma=10, K=10,100,1000, forced x0")
out(f"x0 = [{fmt(x0)}], g(x0) = {tardiness(x0, job_info)}")
res31 = {}
for K in (10, 100, 1000): 
    xb, gb, hist = run(f"3_1_rejection_K{K}.txt", K=K, gamma=10, L=20, x0=x0)
    res31[K] = hist
    out(f"K={K:5d}: best tardiness {gb}, iterations run {len(hist)-1}, best schedule [{fmt(xb)}]")
    out("   new-best iterations: " + ", ".join(f"k={h['k']}:{h['g_best']}" for i, h in enumerate(hist)
                                              if i > 0 and h['g_best'] < hist[i-1]['g_best']))

=== 3.1 Rejection method, L=20, gamma=10, K=10,100,1000, forced x0
x0 = [1,2,3,6,13,5,12,17,7,4,10,14,9,8,18,21,11,16,15,20,19,22,24,23,25,26,27,28,29,30,31], g(x0) = 341
K=   10: best tardiness 335, iterations run 10, best schedule [1,3,6,13,5,12,17,7,4,10,14,2,9,8,18,21,11,16,15,20,19,22,24,23,25,26,27,28,29,30,31]
   new-best iterations: k=10:335
K=  100: best tardiness 214, iterations run 100, best schedule [1,5,7,3,10,14,12,2,8,6,13,18,16,9,4,11,20,21,23,22,26,15,24,28,19,25,17,27,29,30,31]
   new-best iterations: k=10:335, k=15:333, k=16:323, k=19:305, k=20:300, k=38:290, k=39:272, k=47:267, k=50:264, k=52:259, k=53:249, k=61:244, k=64:243, k=68:235, k=69:229, k=70:224, k=75:222, k=87:220, k=89:218, k=94:217, k=98:214
K= 1000: best tardiness 188, iterations run 1000, best schedule [1,7,6,13,14,2,3,5,8,9,12,4,10,16,18,21,23,11,20,17,22,26,15,19,25,27,29,24,28,30,31]
   new-best iterations: k=10:335, k=15:333, k=16:323, k=19:305, k=20:300, k=38:290, k=39:272, k=47:267, k=50:264, k=

## 3.2

In [27]:
out("\n=== 3.2 Rejection method, K=1000, grid over gamma and L (forced x0)")
gammas = [0, 1, 2, 5, 10, 20, 30, 50, 100]; Ls = [1, 5, 10, 20, 30, 50, 100]
# full trace of every (gamma, L) combination -> outputs/3_2_grid/gamma{g}_L{L}.txt
grid = {(g, L): run(f"3_2_grid/gamma{g}_L{L}.txt", gamma=g, L=L, x0=x0) for L in Ls for g in gammas}
out("L \\ gamma " + "".join(f"{g:>7}" for g in gammas))
for L in Ls:
    out(f"{L:>9} " + "".join(f"{grid[(g, L)][1]:>7}" for g in gammas))
out("iterations actually run (<1000 means stopped: no admissible neighbour):")
for L in Ls:
    out(f"{L:>9} " + "".join(f"{len(grid[(g, L)][2])-1:>7}" for g in gammas))
bg, bL = min(grid, key=lambda k: (grid[k][1], k[1], k[0]))
out(f"Best: gamma={bg}, L={bL}, tardiness={grid[(bg, bL)][1]}, schedule=[{fmt(grid[(bg, bL)][0])}]")
run(f"3_2_best_gamma{bg}_L{bL}.txt", gamma=bg, L=bL, x0=x0)   # copy of the best run, easy to find



=== 3.2 Rejection method, K=1000, grid over gamma and L (forced x0)
L \ gamma       0      1      2      5     10     20     30     50    100
        1     257    158    158    170    198    260    260    260    260
        5     257    158    158    170    198    260    260    260    260
       10     257    158    158    170    198    260    260    260    260
       20     257    158    158    158    188    196    196    196    196
       30     257    158    158    163    196    194    194    194    194
       50     257    173    173    173    184    184    184    184    184
      100     257    173    173    173    184    184    184    184    184
iterations actually run (<1000 means stopped: no admissible neighbour):
        1      14   1000   1000   1000   1000   1000   1000   1000   1000
        5      14   1000   1000   1000   1000   1000   1000   1000   1000
       10      14   1000   1000   1000   1000   1000   1000   1000   1000
       20      14    255    255    303   1000

([1,
  7,
  14,
  6,
  5,
  13,
  18,
  21,
  3,
  4,
  23,
  11,
  10,
  16,
  20,
  22,
  26,
  2,
  12,
  17,
  8,
  9,
  15,
  19,
  25,
  27,
  29,
  24,
  28,
  30,
  31],
 158,
 [{'k': 0,
   'x': [1,
    2,
    3,
    6,
    13,
    5,
    12,
    17,
    7,
    4,
    10,
    14,
    9,
    8,
    18,
    21,
    11,
    16,
    15,
    20,
    19,
    22,
    24,
    23,
    25,
    26,
    27,
    28,
    29,
    30,
    31],
   'cost': 341,
   'valid': True,
   'g_best': 341},
  {'k': 1,
   'x': [1,
    3,
    2,
    6,
    13,
    5,
    12,
    17,
    7,
    4,
    10,
    14,
    9,
    8,
    18,
    21,
    11,
    16,
    15,
    20,
    19,
    22,
    24,
    23,
    25,
    26,
    27,
    28,
    29,
    30,
    31],
   'cost': 341,
   'valid': True,
   'g_best': 341},
  {'k': 2,
   'x': [1,
    3,
    6,
    2,
    13,
    5,
    12,
    17,
    7,
    4,
    10,
    14,
    9,
    8,
    18,
    21,
    11,
    16,
    15,
    20,
    19,
    22,
    24,
    23,

## 3.3

In [28]:
out("\n=== 3.3 Penalty method, L=20, gamma=10, K=1000, forced x0")
lambdas = [0.5, 1, 2, 5, 10, 12, 15, 18, 20, 25, 50, 100]; pen = {}
out(f"{'lambda':>7} {'best feasible T':>16} {'% iters infeasible':>19} {'iters run':>10}")
for om in lambdas:
    xb, gb, hist = run(f"3_3_penalty_lambda{om}.txt", gamma=10, L=20, x0=x0, method="penalty", lmbda=om)
    pen[om] = hist
    frac = sum(not h['valid'] for h in hist[1:]) / max(1, len(hist) - 1)
    out(f"{om:>7} {gb:>16} {100*frac:>18.1f}% {len(hist)-1:>10}")


=== 3.3 Penalty method, L=20, gamma=10, K=1000, forced x0
 lambda  best feasible T  % iters infeasible  iters run
    0.5              341              100.0%       1000
      1              341              100.0%       1000
      2              341              100.0%       1000
      5              341              100.0%       1000
     10              300               98.0%       1000
     12              300               98.0%       1000
     15              184               24.9%       1000
     18              179                2.5%       1000
     20              179                2.5%       1000
     25              188                0.0%       1000
     50              188                0.0%       1000
    100              188                0.0%       1000


## Plots

In [29]:
plt.figure(figsize=(7, 4.2))
curve = lambda hist: ([h['k'] for h in hist], [h['g_best'] for h in hist])
plt.step(*curve(res31[1000]), where="post", label="rejection", lw=5, alpha=.35, color="k")
for om in PLOT_LAMBDAS:
    plt.step(*curve(pen[om]), where="post", label=f"penalty, λ={om}", ls="--" if om == PLOT_LAMBDAS[-1] else "-")
plt.xlabel("iteration k"); plt.ylabel("best total tardiness (feasible schedules)")
plt.title("Tabu search, L=20, γ=10, K=1000, same x0"); plt.grid(alpha=.3); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT, "3_3_convergence.png"), dpi=150)

edd = sorted(range(1, len(d) + 1), key=lambda j: (d[j-1], j))   # EDD, ties -> lowest index
out(f"\nEDD x0 = [{fmt(edd)}], g = {tardiness(edd, job_info)}, "
    f"feasible = {check_precedence(edd, prec, 0, job_info)[0]}")
for om in lambdas:
    xb, gb, hist = run(f"3_3_penalty_EDD_lambda{om}.txt", gamma=10, L=20, x0=edd, method="penalty", lmbda=om)
    first = next((h['k'] for h in hist if h['valid']), None)
    frac = sum(not h['valid'] for h in hist[1:]) / max(1, len(hist) - 1)
    out(f"  lambda={om:>6}: first feasible visited at k={first}, returned T={gb}"
        f"{' (repaired, no feasible schedule visited)' if first is None else ''}, "
        f"% infeasible={100*frac:.1f}%, iters={len(hist)-1}")
summary.close()


EDD x0 = [14,26,7,16,23,5,20,29,11,27,4,25,8,30,10,19,15,21,22,28,9,31,12,6,13,18,2,17,1,3,24], g = 0, feasible = False
  lambda=   0.5: first feasible visited at k=None, returned T=315 (repaired, no feasible schedule visited), % infeasible=100.0%, iters=1000
  lambda=     1: first feasible visited at k=None, returned T=315 (repaired, no feasible schedule visited), % infeasible=100.0%, iters=1000
  lambda=     2: first feasible visited at k=None, returned T=232 (repaired, no feasible schedule visited), % infeasible=100.0%, iters=1000
  lambda=     5: first feasible visited at k=None, returned T=198 (repaired, no feasible schedule visited), % infeasible=100.0%, iters=1000
  lambda=    10: first feasible visited at k=None, returned T=256 (repaired, no feasible schedule visited), % infeasible=100.0%, iters=1000
  lambda=    12: first feasible visited at k=636, returned T=232, % infeasible=98.2%, iters=1000
  lambda=    15: first feasible visited at k=619, returned T=189, % infeasible=70.